# What is Multiple Linear Regression?

Multiple Linear Regression is a statistical technique to predict a target variable (the output) using multiple input features (predictors). 

**Simple explanation:** Imagine you want to predict house prices. Instead of using just one factor like size, you use multiple factors like size, location, age, and number of rooms together to make a better prediction. The model learns how much each factor contributes to the final price and creates an equation like:

**Price = (coefficient₁ × size) + (coefficient₂ × location) + (coefficient₃ × age) + ... + constant**

The model finds the best coefficients that minimize the error between predicted and actual values.

# Simple Linear vs Multiple Linear Regression

| Aspect | Simple Linear Regression | Multiple Linear Regression |
|--------|---------------------------|---------------------------|
| **Number of Features** | 1 input variable | 2 or more input variables |
| **Equation** | y = mx + b | y = b₀ + b₁x₁ + b₂x₂ + ... + bₙxₙ |
| **Graph** | 2D line (straight line on a 2D plot) | 3D or higher dimensional (our example uses 2 features, so 3D plane) |
| **Complexity** | Simpler, easier to interpret | More complex but more powerful |
| **Accuracy** | Lower (limited by using only 1 predictor) | Higher (uses multiple predictors for better predictions) |
| **Use Case** | Predicting salary from years of experience | Predicting house price from size, location, age, rooms, etc. |

**In this notebook:** We're building a Multiple Linear Regression model with 2 input features to predict a target value.

In [1]:
from sklearn.datasets import make_regression
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

In [2]:
#  n_samples=100: Create 100 data points
#  n_features=2: Each data point has 2 input features (independent variables)
#  n_informative=2: Both features are informative (they actually affect the target)
#  noise=50: Add some random noise(outliars) to make it more realistic
X, y = make_regression(n_samples=100, n_features=2, n_informative=2, n_targets=1, noise=50)

In [3]:
# Creating dataframe for applying pandas, it works with only datafeames
df = pd.DataFrame({'feature1':X[:, 0], 'feature2':X[:, 1], 'target':y})

In [4]:
df.head()

,feature1,feature2,target
0,-0.095515,-0.040230,-3.020270
1,1.140662,0.130948,-1.487615
2,-0.926311,-0.871469,-96.392443
3,1.374401,1.017856,180.132660
4,-0.381286,0.259166,-12.728017


In [5]:
df.shape

(100, 3)

In [6]:
# Now here we wanna  visualiaze dataset in 3D space
# x-axis: Feature 1 (first input variable)
# y-axis: Feature 2 (second input variable)
# z-axis: Target (what we want to predict)
fig = px.scatter_3d(df, x='feature1', y='feature2', z='target')

fig.show()

In [7]:
# Splitting dataset
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=3)

In [8]:
from sklearn.linear_model import LinearRegression

In [9]:
# Creating model object
lr = LinearRegression()

In [10]:
lr.fit(X_train,y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [11]:
y_pred = lr.predict(X_test)

In [12]:
print("MAE",mean_absolute_error(y_test,y_pred))
print("MSE",mean_squared_error(y_test,y_pred))
print("R2",r2_score(y_test,y_pred))

MAE 54.3223787036114
MSE 4526.747605849738
R2 0.4697326724821502


In [13]:
# Create a mesh grid to generate predictions across the entire 2D feature space
# This helps us visualize the fitted regression plane

# Generate evenly spaced values for both features (-5 to 5)
x = np.linspace(-5, 5, 10)
y = np.linspace(-5, 5, 10)

# Create 2D grid from these values (10x10 = 100 points)
xGrid, yGrid = np.meshgrid(y, x)

# Flatten and combine the grid points into format needed for prediction (100x2)
final = np.vstack((xGrid.ravel().reshape(1, 100), yGrid.ravel().reshape(1, 100))).T

# Predict target values for all points in the grid using our trained model
z_final = lr.predict(final).reshape(10, 10)

# Store predictions in z variable (this will be our regression surface)
z = z_final

In [14]:
# Create comprehensive visualization of multiple linear regression

# 1. Plot the original training data as a scatter plot
# Shows actual observed values (data points) in 3D space
fig = px.scatter_3d(df, x='feature1', y='feature2', z='target')

# 2. Add the fitted regression plane/surface on top
# This is the equation the model learned: it shows the predicted relationship
# between the two input features and the target value
# The plane represents: y = b₀ + b₁(feature1) + b₂(feature2)
fig.add_trace(go.Surface(x=x, y=y, z=z))

# Display the combined visualization
# You can see how well the plane fits through the scattered data points
fig.show()

In [15]:
# These show how much each input feature contributes to predicting the target
# Larger coefficient = stronger influence on the target value
lr.coef_

array([39.05399907, 63.79631723])

In [16]:
# This is the base value when all input features are zero
# The full equation is: target = intercept + (coef1 × feature1) + (coef2 × feature2)
lr.intercept_

np.float64(3.291360116381115)